# Pipeline Project

You will be using the provided data to create a machine learning model pipeline.

You must handle the data appropriately in your pipeline to predict whether an
item is recommended by a customer based on their review.
Note the data includes numerical, categorical, and text data.

You should ensure you properly train and evaluate your model.

## The Data

The dataset has been anonymized and cleaned of missing values.

There are 8 features for to use to predict whether a customer recommends or does
not recommend a product.
The `Recommended IND` column gives whether a customer recommends the product
where `1` is recommended and a `0` is not recommended.
This is your model's target/

The features can be summarized as the following:

- **Clothing ID**: Integer Categorical variable that refers to the specific piece being reviewed.
- **Age**: Positive Integer variable of the reviewers age.
- **Title**: String variable for the title of the review.
- **Review Text**: String variable for the review body.
- **Positive Feedback Count**: Positive Integer documenting the number of other customers who found this review positive.
- **Division Name**: Categorical name of the product high level division.
- **Department Name**: Categorical name of the product department name.
- **Class Name**: Categorical name of the product class name.

The target:
- **Recommended IND**: Binary variable stating where the customer recommends the product where 1 is recommended, 0 is not recommended.

## Load Data

In [ ]:
import pandas as pd

# Load data
df = pd.read_csv(
    'data/reviews.csv',
)

df.info()
df.head()

## Preparing features (`X`) & target (`y`)

In [ ]:
data = df

# separate features from labels
X = data.drop('Recommended IND', axis=1)
y = data['Recommended IND'].copy()

print('Labels:', y.unique())
print('Features:')
display(X.head())

In [3]:
# Split data into train and test sets
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.1,
    shuffle=True,
    random_state=27,
)

# Your Work

## Data Exploration

In [ ]:
# ---- Quick EDA ----
import matplotlib.pyplot as plt
import seaborn as sns

print("Shape:", df.shape)
print("Class balance (1 = recommends):")
print(y.value_counts(normalize=True).round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(x="Recommended IND", data=df, ax=axes[0])
axes[0].set_title("Class balance")
sns.histplot(df["Age"], bins=30, ax=axes[1])
axes[1].set_title("Reviewer age distribution")
plt.tight_layout()
plt.show()

# Departments × class
disp = pd.crosstab(df["Department Name"], df["Recommended IND"], normalize="index")
print("\nP(recommend) by department:")
print(disp.round(3))


## Building Pipeline

In [ ]:
# ---- ColumnTransformer + Pipeline ----
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import GradientBoostingClassifier

NUMERIC = ["Age", "Positive Feedback Count"]
CATEGORICAL = ["Division Name", "Department Name", "Class Name", "Clothing ID"]
TEXT_TITLE = "Title"
TEXT_REVIEW = "Review Text"

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("ohe",     OneHotEncoder(handle_unknown="ignore", min_frequency=20)),
])
title_pipe   = TfidfVectorizer(ngram_range=(1, 2), max_features=400)
review_pipe  = TfidfVectorizer(ngram_range=(1, 2), max_features=2000,
                               stop_words="english")

# Each text vectoriser needs a 1-D Series, hence the FunctionTransformer wrappers.
from sklearn.preprocessing import FunctionTransformer
def column_selector(col):
    return FunctionTransformer(lambda X: X[col].fillna("").values, validate=False)

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe,     NUMERIC),
        ("cat", categorical_pipe, CATEGORICAL),
        ("title",  Pipeline([("sel", column_selector(TEXT_TITLE)),  ("tfidf", title_pipe)]),  []),
        ("review", Pipeline([("sel", column_selector(TEXT_REVIEW)), ("tfidf", review_pipe)]), []),
    ],
    remainder="drop",
)

pipe = Pipeline([
    ("preprocess", preprocess),
    ("clf",        GradientBoostingClassifier(random_state=27)),
])
print(pipe)


## Training Pipeline

In [ ]:
# ---- Fit + evaluate baseline ----
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)
y_prob = pipe.predict_proba(X_test)[:, 1]

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("ROC-AUC:",  round(roc_auc_score(y_test,  y_prob), 4))
print(classification_report(y_test, y_pred, digits=3))


## Fine-Tuning Pipeline

In [ ]:
# ---- Grid search over a few hyper-parameters ----
from sklearn.model_selection import GridSearchCV

grid = {
    "clf__n_estimators": [100, 200],
    "clf__max_depth":     [2, 3],
    "clf__learning_rate": [0.05, 0.1],
}
search = GridSearchCV(pipe, grid, cv=3, n_jobs=-1, scoring="roc_auc", verbose=1)
search.fit(X_train, y_train)
print("Best params:", search.best_params_)
print("Best CV AUC:", round(search.best_score_, 4))

best = search.best_estimator_
y_prob = best.predict_proba(X_test)[:, 1]
print("Held-out AUC:", round(roc_auc_score(y_test, y_prob), 4))
